# Project B - Context-Gap Distillation

KL between the model's predictions with and without the skill document in context, used
as both the importance signal and the distillation loss.

More session-choppable than Project A: each skill-category adapter is self-contained and
`distill/train.py --resume` skips groups already trained. Budget ~5 GPU-hours.

In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'   # cleared only if you prefer a Kaggle Dataset
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

RUNS = '/kaggle/working/artifacts/runs_skill'
SCORES = f'{RUNS}/scores_span.jsonl'

In [ ]:
ARCHIVE = '/kaggle/input/myrios-runs-skill/runs_skill.zip'
run(f"python scripts/kaggle_sync.py restore --archive {ARCHIVE} --run-root {RUNS}")
run(f"python skills/generate_toy_skills.py")


## Gate verification

The riskiest part of the pipeline. High-KL spans must be API identifiers, error codes and
rule clauses; low-KL spans must be markdown scaffolding. Inspect this by eye before
trusting anything trained on it.

In [ ]:
run(f"python kl_gate/score.py --config configs/skill_base.yaml --granularity span --out {SCORES}")
run(f"python kl_gate/inspect_gate.py --scores {SCORES} --top-frac 0.25 --show 6 --out {RUNS}/gate_report.json")


## Distillation and evaluation

`random` is the control that matters: same active-token budget as the KL gate, spans
chosen at random. Any gap between it and `kl_top` is the gate doing real work.

In [ ]:
run(f"python scripts/run_skills.py --config configs/skill_base.yaml --stages distill,eval,report")
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_skill.zip")


In [ ]:
run(f"python scripts/sweep_kl.py --config configs/skill_base.yaml --fracs 0.1,0.25,0.5 --granularities span,token")
run(f"python eval/skill_report.py --report-dir {RUNS}/report --run-root {RUNS} --out docs/results_skills.md")


In [ ]:
from IPython.display import Image, display
display(Image(f'{RUNS}/report/figures/headline_skills.png'))
display(Image(f'{RUNS}/report/figures/kl_threshold_sweep.png'))
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_skill.zip")
